# Project 20: Stock Price Prediction
**Team No.:** G-11  
**Team Members:** Nihar Ranjan Mohapatra; Manoj Kumar Sahu; Debidutta Mahapatra; Chitta Ranjan Mohalik  
**Proposed Hybrid Model:** 1D-CNN + Temporal Transformer  
**Dataset:** NIFTY 50 historical + summary statistics CSVs  
**Source:** https://www.kaggle.com/datasets/kalyan197/nifty50-stocks1999-2026-daily-ohlcv-and-fundamentals


## 0. Setup — Environment, Imports, Reproducibility


In [ ]:
!pip -q install kaggle tqdm tabulate
import os, json, random, glob, subprocess, sys, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print("Using device:", DEVICE)


### CONFIG


In [ ]:
CONFIG = {
 "project_no":"20", "project_name":"Stock Price Prediction", "team_no":"G-11",
 "task_type":"regression", "kaggle_dataset_slug":"kalyan197/nifty50-stocks1999-2026-daily-ohlcv-and-fundamentals",
 "target_column":None, "id_columns":[], "time_column":None,
 "split_ratios":{"train":0.70,"val":0.15,"test":0.15}, "random_seed":SEED,
 "data_raw_dir":"data/20/raw", "data_processed_dir":"data/20/processed", "figures_dir":"data/20/figures", "results_dir":"data/20/results",
 "reports_dir":"data/20/reports", "batch_size":64, "epochs":30, "patience":5
}
for key in ["data_raw_dir","data_processed_dir","figures_dir","results_dir","reports_dir"]: os.makedirs(CONFIG[key],exist_ok=True)
CONFIG


## 1. Dataset Download


In [ ]:
raw=Path(CONFIG["data_raw_dir"])
    subprocess.run(["kaggle","datasets","download","-d",CONFIG["kaggle_dataset_slug"],"-p",str(raw),"--unzip"],check=True)
for z in raw.rglob("*.zip"):
    import zipfile
    with zipfile.ZipFile(z) as f: f.extractall(z.parent/z.stem)
raw_files=[p for p in raw.rglob("*") if p.is_file()]
assert raw_files, "Dataset download produced no files. Configure Kaggle credentials in Colab and rerun."
assert sum(p.stat().st_size for p in raw_files)>1024, "Downloaded content is unexpectedly small."
print(f"Discovered {len(raw_files)} files; {sum(p.stat().st_size for p in raw_files)/2**20:.1f} MiB")


## 2. Load Raw Data


In [ ]:
csvs=list(Path(CONFIG["data_raw_dir"]).rglob("*.csv")); assert csvs,"No CSV files found"
DATE_KEYS=["date","timestamp","datetime","trade_date","dt"]
CLOSE_KEYS=["adj close","adj_close","adjclose","close_price","closing_price","close","last","settle_price","settle"]
def find_col(cols, exact_keys):
    for x in exact_keys:
        if x in cols: return cols[x]
    for norm,orig in cols.items():
        if any(k in norm for k in exact_keys): return orig
    return None
frames=[]
seen_cols={}
for f in csvs:
    try:
        q=pd.read_csv(f); cols={c.lower().strip():c for c in q.columns}
        seen_cols[str(f)]=list(q.columns)
        date=find_col(cols, DATE_KEYS); close=find_col(cols, CLOSE_KEYS)
        if date and close:
            t=q[[date,close]].copy(); t.columns=["date","close"]; t["symbol"]=f.stem; frames.append(t)
    except Exception as e: print("Skipped",f,e)
assert frames, "No CSV with date and close/adjusted-close columns found. Columns seen: " + json.dumps(seen_cols, indent=2)
df=pd.concat(frames,ignore_index=True); df["date"]=pd.to_datetime(df["date"],errors="coerce"); df["close"]=pd.to_numeric(df["close"],errors="coerce"); df=df.dropna().sort_values(["symbol","date"])
chosen=df.groupby("symbol").size().idxmax(); df=df[df.symbol==chosen].drop_duplicates("date").sort_values("date"); series=df.close.to_numpy(dtype="float32")[:,None]
assert len(series)>200; CONFIG["time_column"]="date"; print(chosen,df.date.min(),df.date.max(),len(df))


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo


In [ ]:
missing=float(np.isnan(series).mean())
memo = "\n".join([
    "# Data quality memo",
    f"- Matrix shape: {series.shape}.",
    f"- Missing fraction: {missing:.4f}.",
    "- Chronological splitting prevents future leakage.",
    "- Scaling is fit on the training interval only.",
])
Path("reports/data_quality_memo.md").write_text(memo,encoding="utf-8")


## 4. Preprocessing & Feature Engineering


Feature construction is performed after splitting; every learned imputer, scaler, encoder, graph, and vocabulary is fit on training data only.


## 5. Train / Validation / Test Split


In [ ]:
n=len(series); ntr=int(.70*n); nv=int(.15*n)
raw_series=np.asarray(series,dtype="float32").copy()
# Causal preprocessing: training medians are the only fitted fill values; forward fill uses past observations only.
train_median=np.nanmedian(raw_series[:ntr],axis=0)
causal=pd.DataFrame(raw_series).ffill().to_numpy(dtype="float32")
causal=np.where(np.isnan(causal),train_median,causal).astype("float32")
assert np.isfinite(causal).all()
scaler=StandardScaler().fit(causal[:ntr]); scaled=scaler.transform(causal).astype("float32"); LOOKBACK=24; HORIZON=1
def windows(a,start,end):
    xs=[]; ys=[]
    for t in range(max(start,LOOKBACK),end-HORIZON+1): xs.append(a[t-LOOKBACK:t]); ys.append(a[t:t+HORIZON].squeeze(0))
    return np.stack(xs),np.stack(ys)
Xtr,ytr=windows(scaled,0,ntr); Xv,yv=windows(scaled,ntr,ntr+nv); Xte,yte=windows(scaled,ntr+nv,n)
assert Xtr.shape[0] and Xv.shape[0] and Xte.shape[0]
assert ntr < ntr+nv < n
Path("data/processed/split_manifest.json").write_text(json.dumps({"train":len(Xtr),"val":len(Xv),"test":len(Xte),"lookback":LOOKBACK,"imputation":"training-median plus causal forward-fill"},indent=2),encoding="utf-8")


## 6. PyTorch Dataset & DataLoader


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self,x,y): self.x=torch.tensor(x); self.y=torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]
train_loader=DataLoader(SequenceDataset(Xtr,ytr),CONFIG["batch_size"],shuffle=True); val_loader=DataLoader(SequenceDataset(Xv,yv),CONFIG["batch_size"]); test_loader=DataLoader(SequenceDataset(Xte,yte),CONFIG["batch_size"])


## 7. Proposed Model Definition


In [ ]:
class CNNTemporalTransformer(nn.Module):

    def __init__(self, n, h=64):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(n, h, 3, padding=1), nn.GELU(), nn.Conv1d(h, h, 3, padding=1), nn.GELU())
        enc = nn.TransformerEncoderLayer(h, 4, 128, batch_first=True)
        self.temporal = nn.TransformerEncoder(enc, 2)
        self.head = nn.Linear(h, n)

    def forward(self, x):
        return self.head(self.temporal(self.cnn(x.transpose(1, 2)).transpose(1, 2))[:, -1])


## 8. Training Loop


In [ ]:
from tqdm.auto import tqdm

def run_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None)
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = (xb.to(DEVICE), yb.to(DEVICE))
        if optimizer:
            optimizer.zero_grad(set_to_none=True)
        out = model(xb)
        loss = criterion(out, yb)
        if optimizer:
            loss.backward()
            optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / max(n, 1)

def train_model(model, train_loader, val_loader, checkpoint, classification=False):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss() if classification else nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    history = {'train_loss': [], 'val_loss': []}
    best = float('inf')
    stale = 0
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        tr = run_epoch(model, train_loader, criterion, opt)
        with torch.no_grad():
            va = run_epoch(model, val_loader, criterion)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        scheduler.step(va)
        print(f'epoch={epoch + 1:02d} train={tr:.5f} val={va:.5f}')
        if va < best:
            best = va
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return (model, history)

def predict(model, loader, classification=False):
    model.eval()
    pred = []
    true = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE)).cpu()
            pred.append(torch.softmax(out, 1) if classification else out)
            true.append(yb)
    return (torch.cat(pred).numpy(), torch.cat(true).numpy())
hybrid = CNNTemporalTransformer(series.shape[1])
hybrid, hybrid_history = train_model(hybrid, train_loader, val_loader, 'results/best_hybrid.pt')


## 9. Evaluation Metrics


In [ ]:
results = {}
cached = {}
last_close = scaler.inverse_transform(Xte[:, -1, :])
for name, model in [('hybrid', hybrid)]:
    pred_scaled, y_scaled = predict(model, test_loader)
    pred = scaler.inverse_transform(pred_scaled.reshape(-1, series.shape[1])).reshape(pred_scaled.shape)
    y = scaler.inverse_transform(y_scaled.reshape(-1, series.shape[1])).reshape(y_scaled.shape)
    cached[name] = (pred, y)
    pred_return = (pred - last_close) / (np.abs(last_close) + 1e-08)
    true_return = (y - last_close) / (np.abs(last_close) + 1e-08)
    results[name] = {'mae_price_units': mean_absolute_error(y, pred), 'rmse_price_units': mean_squared_error(y, pred) ** 0.5, 'r2': r2_score(y, pred, multioutput='variance_weighted'), 'directional_accuracy': float((np.sign(pred_return) == np.sign(true_return)).mean()), 'return_mae': mean_absolute_error(true_return, pred_return)}
Path('results/metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
results


## 10. Required Figures


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hybrid_history['train_loss'], label='hybrid train')
plt.plot(hybrid_history['val_loss'], label='hybrid val')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    probs, pred, y = cached['hybrid']
    cm = confusion_matrix(y, pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('figures/fig02_confusion_matrix.png', dpi=150)
    plt.show()
    support = np.bincount(y, minlength=probs.shape[1])
    per_class = [(pred[y == i] == i).mean() if (y == i).any() else np.nan for i in range(probs.shape[1])]
    plt.figure(figsize=(9, 4))
    plt.bar(range(len(per_class)), per_class)
    plt.ylabel('Per-class recall')
    plt.tight_layout()
    plt.savefig('figures/fig03_class_performance.png', dpi=150)
    plt.show()
else:
    pred, y = cached['hybrid']
    plt.figure(figsize=(6, 5))
    plt.scatter(y.ravel(), pred.ravel(), s=8, alpha=0.35)
    lo = min(y.min(), pred.min())
    hi = max(y.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], 'k--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.tight_layout()
    plt.savefig('figures/fig02_predicted_vs_actual.png', dpi=150)
    plt.show()
    residual = (pred - y).ravel()
    plt.figure(figsize=(7, 4))
    sns.histplot(residual, bins=40, kde=True)
    plt.xlabel('Residual')
    plt.tight_layout()
    plt.savefig('figures/fig03_residual_distribution.png', dpi=150)
    plt.show()
xb, yb = next(iter(test_loader))
xb = xb[:min(32, len(xb))].to(DEVICE).requires_grad_(True)
hybrid.zero_grad()
out = hybrid(xb)
score = out.max(1).values.sum() if out.ndim == 2 and out.shape[1] > 1 else out.sum()
score.backward()
importance = xb.grad.detach().abs().cpu().numpy()
imp = importance.mean(axis=tuple(range(importance.ndim - 1))) if importance.ndim > 2 else importance.mean(0)
plt.figure(figsize=(8, 4))
plt.plot(np.ravel(imp))
plt.title('Gradient-based input importance')
plt.tight_layout()
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    errors = (cached['hybrid'][1] != cached['hybrid'][2]).astype(int)
else:
    errors = np.abs(cached['hybrid'][0] - cached['hybrid'][1]).reshape(len(cached['hybrid'][1]), -1).mean(1)
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30)
plt.title('Held-out error distribution')
plt.tight_layout()
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
metric = next(iter(results['hybrid']))
plt.figure(figsize=(6, 4))
plt.bar(results.keys(), [results[k][metric] for k in results])
plt.ylabel(metric)
plt.tight_layout()
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
